# Día 10 · Ejercicios — ML aplicado y actuarial-financiero cuantitativo

**Duración estimada:** 90-120 minutos.

Cuatro ejercicios: dos de la Unidad 11 (ML aplicado, sobre `02_teoria_pipelines_riesgos_ml.ipynb`) y dos de la Unidad 12 (renta y provisiones, sobre `04_teoria_rentas.ipynb` y `07_teoria_provisiones_chain_ladder.ipynb`). Cada ejercicio es independiente: puedes hacerlos en cualquier orden.

Las soluciones están en `09_soluciones.ipynb` — inténtalo primero por tu cuenta.

## Ejercicio 1 (Unidad 11) — Otro clasificador dentro del mismo Pipeline

En `02_teoria_pipelines_riesgos_ml.ipynb` comparamos `LogisticRegression` y `RandomForestClassifier` para predecir `anulada` en `data/raw/polizas_auto_anulacion.csv`. Ahora te toca añadir un tercer candidato: `GradientBoostingClassifier`.

1. Carga `../data/raw/polizas_auto_anulacion.csv`.
2. Construye el mismo `ColumnTransformer` que en la teoría (imputación + `RobustScaler` en numéricas, `OneHotEncoder` en categóricas, `passthrough` en la booleana). Puedes copiar las listas de columnas de `02_teoria_pipelines_riesgos_ml.ipynb`.
3. Crea un `Pipeline` con ese preprocesador y un `GradientBoostingClassifier(random_state=42)` como modelo (nota: `GradientBoostingClassifier` no admite `class_weight` — es una diferencia frente a los dos modelos de la teoría, que sí lo admitían).
4. Divide en train/test con `train_test_split(..., stratify=y, random_state=42, test_size=0.25)`.
5. Entrena y calcula `precision`, `recall` y `f1` de la clase `anulada` en el conjunto de test.
6. Compara tu `f1` con el del `RandomForestClassifier` ajustado de la teoría (~0.22 en aquel `random_state`). ¿Mejora, empeora, o es parecido?

In [ ]:
import numpy as np
import pandas as pd

# 1. Carga el dataset
# tu codigo aqui


In [ ]:
# 2-3. ColumnTransformer + Pipeline con GradientBoostingClassifier
# tu codigo aqui


In [ ]:
# 4-5. train_test_split, entrenamiento y metricas
# tu codigo aqui


**Pregunta para responder en una celda markdown**: ¿qué modelo elegirías para este problema, y por qué — solo por el `f1`, o hay algo más (interpretabilidad, tiempo de entrenamiento, riesgo de sobreajuste) que también pesaría en tu decisión?

In [ ]:
# tu respuesta (markdown o comentario)


## Ejercicio 2 (Unidad 11) — Encuentra la fuga de datos

Un compañero de curso ha escrito este código para seleccionar las variables numéricas más relevantes **antes** de entrenar un modelo de anulación de pólizas, para "simplificar" el dataset. Ejecuta la celda tal cual está (usa el propio `polizas_auto_anulacion.csv`) y luego responde a las preguntas — **no corrijas nada todavía**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

polizas = pd.read_csv("../data/raw/polizas_auto_anulacion.csv")
columnas_numericas = [
    "edad_conductor", "antiguedad_carnet", "antiguedad_vehiculo", "potencia_cv",
    "km_anuales", "num_siniestros_3_anios", "antiguedad_poliza_meses", "prima_anual",
]

datos = polizas.copy()
datos["anulada_num"] = datos["anulada"].astype(int)
datos[columnas_numericas] = datos[columnas_numericas].fillna(datos[columnas_numericas].median())

# "Seleccion de variables": nos quedamos solo con las 4 numericas mas
# correlacionadas (en valor absoluto) con la variable objetivo...
# calculado sobre TODO el dataset, antes de dividir en train/test.
correlaciones = datos[columnas_numericas + ["anulada_num"]].corr()["anulada_num"].drop("anulada_num")
mejores_columnas = correlaciones.abs().sort_values(ascending=False).head(4).index.tolist()
print("Columnas seleccionadas:", mejores_columnas)

X = datos[mejores_columnas]
y = datos["anulada"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
modelo = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo.fit(X_train, y_train)
print(f"f1 en test: {f1_score(y_test, modelo.predict(X_test)):.3f}")

**Preguntas:**

1. ¿Dónde está exactamente la fuga de datos en este código? Señala la línea concreta.
2. ¿Qué información "ve" el modelo durante la selección de variables que no debería ver hasta después de entrenar?
3. Reescribe el flujo para que la selección de variables se calcule **solo con el conjunto de entrenamiento**, y vuelve a calcular el `f1`. ¿Cambian las columnas seleccionadas? ¿Cambia mucho el `f1`?

In [ ]:
# tu respuesta a las preguntas 1 y 2 (como comentario o celda markdown)


In [ ]:
# 3. version corregida: seleccion de variables solo con el train
# tu codigo aqui


## Ejercicio 3 (Unidad 12) — Valor actual de una indemnización en forma de renta

Una aseguradora va a indemnizar a un asegurado con una renta **constante** de 12.000€/año durante 8 años, con un tipo de descuento del 3,5%.

1. Usando (o reimplementando) `pv_renta_constante` de `04_teoria_rentas.ipynb`, calcula el valor actual de esta renta si el primer pago es **pospagable** (al final del primer año).
2. Calcula también el valor actual si el primer pago es **prepagable** (al inicio del primer año) — pista: es `pv_renta_constante(...) * (1 + i)`.
3. Valida tu resultado del punto 1 con una suma explícita de flujos descontados (`np.sum(...)`), igual que se hace en la teoría.
4. La aseguradora se plantea una alternativa: pagar un único importe hoy (capital único) en vez de la renta. ¿Qué importe único, pagado hoy, sería financieramente equivalente a la renta pospagable del punto 1?

In [ ]:
def pv_renta_constante(C: float, i: float, n: int) -> float:
    """Valor actual de una renta constante, temporal, pospagable."""
    # tu codigo aqui (o copialo de 04_teoria_rentas.ipynb)
    pass


In [ ]:
# 1-2. Renta pospagable y prepagable
# tu codigo aqui


In [ ]:
# 3. Validacion con suma explicita de flujos
# tu codigo aqui


In [ ]:
# 4. Capital unico equivalente
# tu codigo aqui


## Ejercicio 4 (Unidad 12) — Chain Ladder con un triángulo nuevo

Se te da el siguiente triángulo de siniestros pagados acumulados (en miles de EUR), para 4 años de origen:

| año origen \ desarrollo | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| 2021 | 800 | 1250 | 1380 | 1420 |
| 2022 | 900 | 1400 | 1550 | ? |
| 2023 | 950 | 1480 | ? | ? |
| 2024 | 1000 | ? | ? | ? |

1. Construye el `DataFrame` del triángulo (con `NaN` donde corresponda).
2. Calcula los factores de desarrollo volumen-ponderados (puedes reutilizar `calcular_factores_desarrollo` de `07_teoria_provisiones_chain_ladder.ipynb`, o reimplementarla).
3. Proyecta el triángulo completo y calcula la provisión por año de origen y la provisión total.
4. Valida que el año 2021 (totalmente desarrollado) tiene provisión exactamente 0.

In [ ]:
# 1. Construccion del triangulo
# tu codigo aqui


In [ ]:
# 2. Factores de desarrollo
# tu codigo aqui


In [ ]:
# 3. Proyeccion y provision por anio + total
# tu codigo aqui


In [ ]:
# 4. Validacion: provision del 2021 debe ser 0
# tu codigo aqui
